In [1]:
import torch
import os
import glob
from unsloth import FastLanguageModel
import pandas as pd
from datasets import load_dataset
from torch.utils.data import DataLoader

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [2]:
BATCH_SIZE = 4
MAX_INPUT_LENGTH = 1024
MAX_NEW_TOKENS = 512

In [3]:

dataset = load_dataset("juletxara/mgsm", 'bn', split = "train")
test_dataset = load_dataset("juletxara/mgsm", 'bn', split = "test")

In [5]:
df = pd.DataFrame(dataset)
df[:21]


,question,answer,answer_number,equation_solution
0,প্রশ্ন: রজারের 5টি টেনিস বল আছে। সে আরও 2 ক্যা...,ধাপে ধাপে উত্তর: রজারের প্রথমে 5টি বল ছিল। 2টি...,11,5 + 6 = 11.
1,প্রশ্ন: সার্ভার কক্ষে নয়টি কম্পিউটার ছিল। সোমব...,ধাপে ধাপে উত্তর: সোমবার থেকে বৃহস্পতিবার 4দিন ...,29,4 * 5 = 20. 9 + 20 = 29.
2,প্রশ্ন: লিয়ার 32টি চকোলেট ছিল এবং তার বোনের ছি...,ধাপে ধাপে উত্তর: লিয়ার 32টি চকোলেট ছিল এবং লিয়...,39,32 + 42 = 74. 74 - 35 = 39.
3,প্রশ্ন: শনের পাঁচটি খেলনা আছে। ক্রিসমাস উপলক্ষ...,ধাপে ধাপে উত্তর: তার কাছে 5টি খেলনা আছে। সে তা...,9,5 + 2 = 7. 7 + 2 = 9.
4,"প্রশ্ন: মাইকেলের 58টি গলফ বল ছিল। মঙ্গলবার, সে...",ধাপে ধাপে উত্তর: শুরুতে মাইকেলের কাছে 58টি গলফ...,33,58 - 23 = 35. 35 - 2 = 33.
5,প্রশ্ন: অলিভিয়ার $23 আছে। সে পাঁচটি ব্যাগেল কি...,ধাপে ধাপে উত্তর: প্রতিটি ব্যাগেলের মূল্য $3 হল...,8,5 * 3 = 15. 23 - 15 = 8.
6,প্রশ্ন: জ্যাসনের 20টি ললিপপ ছিল। সে ডেনিকে কিছ...,ধাপে ধাপে উত্তর: জ্যাসনের কাছে প্রথমে 20টি ললি...,8,20 - 12 = 8.
7,প্রশ্ন: পার্কিং লটে যদি 3 টি গাড়ি থাকে এবং আরও...,"ধাপে ধাপে উত্তর: শুরুতে সেখানে 3টি গাড়ি ছিল, আ...",5,3 + 2 = 5.


In [7]:
df.to_excel("../../MGSM_bn.xlsx", index=False)

In [8]:
max_seq_length = 1024 # Choose any! Unsloth also supports RoPE (Rotary Positinal Embedding) scaling internally.
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "dipta007/GanitLLM-4B-SFT", 
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit, # Will load the 4Bit Quantized Model
)

==((====))==  Unsloth 2026.7.1: Fast Qwen3 patching. Transformers: 4.57.6.
   \\   /|    NVIDIA GeForce RTX 3060. Num GPUs = 1. Max memory: 11.622 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [28]:
import trl
print(trl.__file__)

/home/iztihad/venvs/ml/lib/python3.12/site-packages/trl/__init__.py


In [3]:
model.eval()

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2560, padding_idx=151643)
    (layers): ModuleList(
      (0-35): 36 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear4bit(in_features=2560, out_features=4096, bias=False)
          (k_proj): Linear4bit(in_features=2560, out_features=1024, bias=False)
          (v_proj): Linear4bit(in_features=2560, out_features=1024, bias=False)
          (o_proj): Linear4bit(in_features=4096, out_features=2560, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear4bit(in_features=2560, out_features=9728, bias=False)
          (up_proj): Linear4bit(in_features=2560, out_features=9728, bias=False)
          (down_proj): Linear4bit(in_features=9728, out_features=2560, bias=False)
          (act_fn): SiLUActivation()
    

In [6]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)


def generate_response(prompt, system_prompt=None):

    messages = []

    if system_prompt:
        messages.append({
            "role": "system",
            "content": system_prompt
        })

    messages.append({
        "role": "user",
        "content": prompt
    })

    # Create input IDs + attention mask
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=2048,
        temperature=0.7,
        min_p=0.1,
        use_cache=True,
    )

    # Remove the input tokens
    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    # Decode only the generated response
    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()

In [ ]:
while True:
    prompt = input("\nYou: ")

    if prompt.lower() in ["exit", "quit"]:
        break

    response = generate_response(prompt)

    print("\nAssistant:", response)
    

In [17]:
def create_prompt(question):

    messages = [
        {
            "role": "user",
            "content": (
                "Solve the following math problem. "
                "Show your reasoning and give the final answer clearly.\n\n"
                f"Question:\n{question}"
            )
        }
    ]

    # Use model's chat template if available
    if tokenizer.chat_template is not None:

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

    else:

        prompt = (
            "Solve the following math problem. "
            "Show your reasoning and give the final answer clearly.\n\n"
            f"Question:\n{question}\n\n"
            "Answer:"
        )

    return prompt

In [18]:
question = test_dataset[0]["question"]

prompt = create_prompt(question)

print(prompt)

<|im_start|>user
Solve the following math problem. Show your reasoning and give the final answer clearly.

Question:
Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?<|im_end|>
<|im_start|>assistant



In [9]:
def collate_fn(batch):

    questions = [
        item["question"]
        for item in batch
    ]

    answers = [
        item["answer_number"]
        for item in batch
    ]

    prompts = [
        create_prompt(q)
        for q in questions
    ]

    return {
        "questions": questions,
        "answers": answers,
        "prompts": prompts
    }



dataloader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

In [ ]:
import re

def extract_mgsm_answer(text):
    """
    Extract the final answer from a GSM8K solution.

    Expected reference format:
        #### 42

    Also has a fallback that extracts the last number.
    """

    # Preferred: GSM8K's #### format
    matches = re.findall(
        r"####\s*(-?\d+(?:\.\d+)?)",
        text
    )

    if matches:
        return matches[-1]

    # Fallback: extract the last number
    matches = re.findall(
        r"-?\d+(?:\.\d+)?",
        text
    )

    if matches:
        return matches[-1]

    return None

In [21]:

CHECKPOINT_EVERY = 10
CHECKPOINT_DIR = "GSM8K_checkpoints"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)


# ============================================================
# Find latest checkpoint
# ============================================================

checkpoint_files = glob.glob(
    os.path.join(
        CHECKPOINT_DIR,
        "checkpoint_batch_*.csv"
    )
)


if checkpoint_files:

    def get_batch_number(path):
        filename = os.path.basename(path)
        match = re.search(
            r"checkpoint_batch_(\d+)\.csv",
            filename
        )
        return int(match.group(1))

    latest_checkpoint = max(
        checkpoint_files,
        key=get_batch_number
    )

    last_batch = get_batch_number(
        latest_checkpoint
    )

    print("Latest checkpoint found:")
    print(latest_checkpoint)
    print("Last completed batch:", last_batch)

else:

    latest_checkpoint = None
    last_batch = 0

    print("No checkpoint found.")
    print("Starting evaluation from the beginning.")

Latest checkpoint found:
GSM8K_checkpoints/checkpoint_batch_330.csv
Last completed batch: 330


In [22]:
if latest_checkpoint is not None:

    checkpoint_df = pd.read_csv(
        latest_checkpoint
    )

    results = checkpoint_df.to_dict(
        orient="records"
    )

    total = len(results)

    correct = sum(
        1 for result in results
        if result["correct"]
    )

    print("Checkpoint loaded.")
    print("Already processed:", total)
    print("Already correct:", correct)

    if total > 0:
        print(
            f"Current accuracy: "
            f"{correct / total * 100:.2f}%"
        )

else:

    results = []
    correct = 0
    total = 0

Checkpoint loaded.
Already processed: 1319
Already correct: 362
Current accuracy: 27.45%


In [ ]:
# Number of examples already processed
start_index = total

print("Starting from example:", start_index)
print("Remaining examples:", len(test_dataset) - start_index)


# Create a subset containing only unprocessed examples
remaining_dataset = test_dataset.select(
    range(start_index, len(test_dataset))
)


dataloader = DataLoader(
    remaining_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)


print("Remaining batches:", len(dataloader))

In [ ]:
for batch_idx, batch in enumerate(dataloader):

    questions = batch["questions"]
    reference_texts = batch["answers"]
    prompts = batch["prompts"]

    # --------------------------------------------------------
    # Tokenize batch
    # --------------------------------------------------------

    inputs = tokenizer(
        prompts,
        padding=True,
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
        return_tensors="pt"
    )

    # Move to GPU
    inputs = {
        key: value.to(model.device)
        for key, value in inputs.items()
    }

    # --------------------------------------------------------
    # Generate
    # --------------------------------------------------------

    with torch.inference_mode():

        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # --------------------------------------------------------
    # Remove input tokens
    # --------------------------------------------------------

    input_length = inputs["input_ids"].shape[1]

    generated_tokens = outputs[:, input_length:]

    predictions = tokenizer.batch_decode(
        generated_tokens,
        skip_special_tokens=True
    )

    # --------------------------------------------------------
    # Evaluate examples
    # --------------------------------------------------------

    for question, reference_text, prediction in zip(
        questions,
        reference_texts,
        predictions
    ):

        predicted_answer = extract_mgsm_answer(
            prediction
        )

        reference_answer = extract_mgsm_answer(
            reference_text
        )

        is_correct = (
            predicted_answer == reference_answer
        )

        if is_correct:
            correct += 1

        total += 1

        results.append({
            "question": question,
            "reference_solution": reference_text,
            "model_output": prediction,
            "reference_answer": reference_answer,
            "predicted_answer": predicted_answer,
            "correct": is_correct
        })

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    current_accuracy = correct / total

    # Overall batch number
    current_batch = last_batch + batch_idx + 1

    print(
        f"Batch {current_batch} | "
        f"Examples: {total}/{len(test_dataset)} | "
        f"Accuracy: {current_accuracy * 100:.2f}%"
    )

    # --------------------------------------------------------
    # Save checkpoint every 10 batches
    # --------------------------------------------------------

    if (batch_idx + 1) % CHECKPOINT_EVERY == 0:

        checkpoint_df = pd.DataFrame(results)

        checkpoint_path = os.path.join(
            CHECKPOINT_DIR,
            f"checkpoint_batch_{current_batch}.csv"
        )

        checkpoint_df.to_csv(
            checkpoint_path,
            index=False
        )

        print()
        print("=" * 60)
        print("CHECKPOINT SAVED")
        print("=" * 60)
        print("File:", checkpoint_path)
        print("Processed:", total)
        print(
            f"Accuracy: "
            f"{current_accuracy * 100:.2f}%"
        )
        print("=" * 60)
        print()

Batch 321 | Examples: 1284/1319 | Accuracy: 27.41%
Batch 322 | Examples: 1288/1319 | Accuracy: 27.41%
Batch 323 | Examples: 1292/1319 | Accuracy: 27.40%
Batch 324 | Examples: 1296/1319 | Accuracy: 27.47%
Batch 325 | Examples: 1300/1319 | Accuracy: 27.38%
Batch 326 | Examples: 1304/1319 | Accuracy: 27.45%
Batch 327 | Examples: 1308/1319 | Accuracy: 27.45%
Batch 328 | Examples: 1312/1319 | Accuracy: 27.44%
Batch 329 | Examples: 1316/1319 | Accuracy: 27.43%
Batch 330 | Examples: 1319/1319 | Accuracy: 27.45%

CHECKPOINT SAVED
File: GSM8K_checkpoints/checkpoint_batch_330.csv
Processed: 1319
Accuracy: 27.45%



In [16]:
accuracy = correct / total

print("=" * 60)
print("FINAL RESULTS")
print("=" * 60)

print(f"Correct : {correct}")
print(f"Total   : {total}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Accuracy: {accuracy * 100:.2f}%")

FINAL RESULTS
Correct : 362
Total   : 1319
Accuracy: 0.2745
Accuracy: 27.45%
